# 5-1 PEFT(파라미터 효율적 튜닝) 실습

---

## 학습 개요

### 학습 주제
- **Full Fine-tuning vs PEFT(파라미터 효율적 튜닝) 비교**: 사전학습된 대형 언어 모델을 새로운 태스크에 적응시키는 두 가지 접근법을 비교하고, 각각의 메모리/연산 요구사항과 성능 trade-off를 분석합니다.
- **LoRA(Low-Rank Adaptation) 원리와 구현**: 가중치 행렬의 저차원 분해를 활용한 효율적인 파인튜닝 기법의 수학적 원리를 이해하고, 실제 코드로 구현합니다.
- **QLoRA를 활용한 효율적인 LLM 파인튜닝**: 양자화(Quantization)와 LoRA를 결합하여 제한된 GPU 메모리 환경에서도 대형 모델을 학습하는 방법을 실습합니다.

### 학습 목표
- LoRA 기반 PEFT를 활용하여 **제한된 GPU 환경에서 LLM 파인튜닝**을 수행하고, **학습 결과를 저장/추론**하는 것을 목표로 합니다.
  - Full Fine-tuning과 PEFT의 차이점을 이해하고 PEFT가 필요한 이유를 **설명**할 수 있다.
  - LoRA의 원리(저차원 분해, Adapter 개념)를 이해하고 핵심 하이퍼파라미터를 **설정**할 수 있다.
  - Unsloth를 활용하여 QLoRA 기반 LLM 파인튜닝을 **수행**하고 결과를 확인할 수 있다.
  - 파인튜닝된 모델을 **저장**하고 추론에 **활용**할 수 있다.
  - 학습된 LoRA adapter의 파라미터 효율성을 **분석**하고 Full Fine-tuning과 **비교**할 수 있다.

### 핵심 개념
- **PEFT(Parameter-Efficient Fine-Tuning)**: 전체 모델 파라미터 중 일부만 학습하여 메모리와 시간을 절약하는 기법입니다. 사전학습된 가중치는 고정(freeze)하고, 소수의 추가 파라미터(adapter)만 학습하여 Full Fine-tuning 대비 10~1000배 적은 파라미터로 유사한 성능을 달성합니다.
- **LoRA(Low-Rank Adaptation)**: 가중치 행렬을 저차원 행렬로 분해하여 학습 파라미터를 크게 줄이는 방법입니다. 원본 가중치 $W \in \mathbb{R}^{d \times k}$에 대해 업데이트를 $\Delta W = BA$로 표현하며, $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$ (rank $r \ll \min(d, k)$)입니다. 최종 출력은 $W' = W + BA$로 계산됩니다.
- **QLoRA(Quantized LoRA)**: 양자화된 모델에 LoRA를 적용하여 더욱 효율적인 학습이 가능합니다. Base model을 4-bit NormalFloat(NF4) 양자화로 로드하고, LoRA adapter는 BFloat16으로 학습하여 16GB VRAM에서 7B 모델도 파인튜닝할 수 있습니다.
- **Unsloth**: LoRA/QLoRA 학습을 최적화한 프레임워크로 학습 속도와 메모리 효율을 향상시킵니다. 수작업으로 최적화된 GPU 커널(Triton 기반)을 사용하여 FlashAttention 2 대비 최대 30배 빠른 학습 속도와 30% 메모리 절감을 달성합니다.
- **SFTTrainer**: HuggingFace TRL 라이브러리의 Supervised Fine-Tuning 트레이너입니다. 데이터 로딩 → 토큰화 → 배치 구성 → 순전파 → 역전파 → 가중치 업데이트의 전체 파이프라인을 자동으로 관리하며, PEFT와의 통합을 지원합니다.

### 선행 지식
- **PyTorch 프로그래밍 (중급)**: 텐서 연산, 모델 정의, 학습 루프 구현 경험 필요
- **Transformer 구조 이해 (기초)**: Self-Attention, Multi-Head Attention, Feed-Forward Network의 역할 이해
- **선형대수 기초**: 행렬 곱셈, 행렬 분해(SVD) 개념 이해
- **선수 학습**: Chapter 2-1 토큰화/임베딩 실습 완료 권장

---

## 실습 구성

### 학습 방향

- **실습 구성 방식**
  - 문제와 정답 코드가 병렬로 제공되며, 각 단계별 TODO 영역을 채우며 학습자가 직접 구현


- **실행 환경**
  - Python 3.10 이상
  - GPU: NVIDIA GPU 권장 (16GB VRAM 이상)

- **Required Package**
  - `unsloth>=2025.8.0`: 최적화된 LLM 파인튜닝 프레임워크
  - `transformers>=4.55.0`: HuggingFace 모델/토크나이저
  - `peft>=0.10.0`: Parameter-Efficient Fine-Tuning 라이브러리
  - `trl>=0.8.0`: Transformer Reinforcement Learning (SFTTrainer)
  - `torch>=2.0.0`: PyTorch 딥러닝 프레임워크
  - `datasets>=3.0.0`: HuggingFace 데이터셋 라이브러리
  - `accelerate>=0.30.0`: 분산 학습 및 혼합 정밀도 지원
  - `bitsandbytes>=0.43.0`: 양자화 및 8-bit 옵티마이저
  - `matplotlib>=3.7.0`: 메모리 시각화 (Full FT 문제 체감)

- **Step 요약**
  - **Step 1 (10분)**: 환경 설정 — 라이브러리 설치 및 모델 로드
  - **Step 2 (10분)**: Full Fine-tuning 시도 — 메모리 부족 문제 체감
  - **Step 3 (10분)**: PEFT/LoRA 원리 이해 — 1% 파라미터만 학습하는 원리 학습
  - **Step 4 (20분)**: LoRA 적용 및 학습 — 16GB VRAM으로 실제 학습 수행
  - **Step 5 (10분)**: 결과 확인 및 저장 — 모델 저장 및 추론 테스트

- **데이터셋 개요 및 저작권 정보**
  - **데이터셋 명**: Thytu/ChessInstruct
  - **데이터셋 개요**: 체스 기보 예측을 위한 instruction following 데이터셋으로, task/input/expected_output 형태로 구성되어 있습니다.
  - **사용 목적**: instruction tuning을 통한 LLM 파인튜닝 실습
  - **저작권/출처**: HuggingFace Hub (MIT License)
  - **주의사항**: 본 실습에서는 학습 시간을 고려하여 전체 데이터셋 중 일부(5,000개)만 사용합니다. max_seq_length 설정에 따라 메모리 사용량이 달라질 수 있습니다.

### 문제 설명

- **문제 개요**: 이 실습은 **"문제 체감 → 해결"** 흐름으로 진행됩니다. 학습자는 먼저 **Full Fine-tuning을 시도하여 메모리 부족 문제를 직접 체감**하고, 이후 **PEFT/LoRA가 왜 필요한지**를 자연스럽게 이해합니다. 최종적으로 **1% 미만의 파라미터만 학습하는 LoRA를 적용**하여 **16GB VRAM 환경에서도 LLM 파인튜닝이 가능함**을 체험합니다.
- **요구사항 요약**
  - **메모리 분석**: Full Fine-tuning에 필요한 메모리를 계산하고 제약 조건 확인
  - **LoRA 설정**: 핵심 하이퍼파라미터(r, lora_alpha, target_modules) 설정
  - **데이터 변환**: 데이터셋을 chat 형식(conversations)으로 변환
  - **추론 테스트**: 학습된 모델로 테스트 입력 구성 및 추론 수행

### 학습 문제

- **Step 2 — Full Fine-tuning 시도**
  - **TODO 1**: Full Fine-tuning 메모리 요구량 계산 및 OOM 체감 *(연결: PEFT 필요성 / 메모리 분석)*
  - **1줄 요약**: 메모리 요구량을 계산하고, 실제 Full FT 시도로 OOM 에러를 체감한다.
- **Step 4 — LoRA 적용 및 학습**
  - **TODO 2**: LoRA 설정값 채우기 *(연결: LoRA / 하이퍼파라미터 설정)*
  - **1줄 요약**: r, lora_alpha, target_modules 등 LoRA 핵심 파라미터를 설정한다.
  - **TODO 3**: convert_to_chatml 함수 작성 *(연결: Chat Template / 데이터 전처리)*
  - **1줄 요약**: 데이터셋을 system/user/assistant 형식의 conversation으로 변환한다.
- **Step 5 — 결과 확인 및 저장**
  - **TODO 4**: 추론을 위한 테스트 입력 구성 *(연결: Chat Template / 추론)*
  - **1줄 요약**: 테스트 메시지를 구성하고 학습된 모델로 추론을 수행한다.

### 주의사항
- GPU 메모리 부족 시 `per_device_train_batch_size`를 줄이거나 `gradient_accumulation_steps`를 늘리세요.
- 학습 중 loss가 발산하면 `learning_rate`를 낮추는 것을 권장합니다.
- 실습 환경에 따라 모델 로딩 시간이 다를 수 있으며, 첫 실행 시 모델 다운로드에 시간이 소요됩니다.


---

# Step 1: 환경 설정

라이브러리를 설치하고 Unsloth의 개념을 이해합니다.

### Setup: 라이브러리 설치

In [ ]:
%%capture
# 로컬 환경용 설치 (RTX 5060ti, 16GB VRAM)
%pip install unsloth
%pip install --upgrade typing_extensions
%pip install "transformers>=4.55.0" "peft>=0.10.0" "trl>=0.8.0"
%pip install "datasets>=3.0.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"
%pip install "matplotlib>=3.7.0"

### Concept Check: Unsloth란?

> **🧠 Unsloth란?**
> Unsloth는 효율적인 LLM(대형 언어 모델) 파인튜닝 및 강화학습(RL) 프레임워크로, 속도와 메모리 관리 측면에서 최적화된 성능을 제공합니다.

**주요 특징:**
- 수작업으로 최적화된 GPU 커널을 사용하여 기존보다 훨씬 빠른 학습 가능
- 오픈소스이며 무료 버전 제공
- 기존 FlashAttention 2 기반보다 최대 **30배 빠른 학습 속도**와 **메모리(VRAM) 30% 절감**
- QLoRA, LoRA, 4-/8-/16-bit 등 저정밀 훈련 지원

**왜 Unsloth를 사용하나요?**
- 16GB VRAM 환경에서도 대형 모델 파인튜닝 가능
- 표준 HuggingFace PEFT 대비 학습 속도 향상
- 간편한 API로 빠른 실험 가능

### Concept Check: LoRA vs QLoRA — 무엇이 다른가?

이 실습에서는 **QLoRA**를 사용합니다. LoRA와 QLoRA의 차이를 이해하면 "지금 하고 있는 것"이 무엇인지 명확해집니다.

**LoRA vs QLoRA 비교:**

| 구분 | LoRA | QLoRA |
|------|------|-------|
| **Base Model** | FP16/BF16 (원본) | 4-bit 양자화 |
| **LoRA Adapter** | FP16/BF16 | FP16/BF16 (동일) |
| **핵심 차이** | 모델 로드 방식 | 모델 로드 방식 |
| **1B 모델 메모리** | ~2GB | ~0.5GB |
| **7B 모델 메모리** | ~14GB | ~4GB |

**QLoRA의 핵심 기술 3가지:**

1. **NF4 (4-bit NormalFloat)**
   - 가중치 분포에 최적화된 4-bit 양자화 방식
   - 일반 INT4보다 정보 손실이 적음

2. **Double Quantization**
   - 양자화 상수까지 다시 양자화하여 메모리 추가 절감
   - 약 0.5GB 추가 절약 (7B 모델 기준)

3. **Paged Optimizers**
   - GPU 메모리 부족 시 CPU 메모리로 자동 스왑
   - OOM 방지 및 더 큰 배치 사이즈 허용

> **핵심**: QLoRA = **Q**(양자화된 Base Model) + **LoRA**(저차원 Adapter)
> 
> 아래에서 모델을 로드할 때 `bnb-4bit`가 바로 **"Q"** 부분입니다!

### Guided Build: 모델 및 토크나이저 로드

Unsloth를 사용하여 4-bit 양자화된 모델을 로드합니다.

> **참고**: `unsloth/gemma-3-1b-it-unsloth-bnb-4bit`는 Unsloth가 파인튜닝에 최적화하여 양자화한 모델입니다.

In [ ]:
from unsloth import FastModel
import torch

# 재현성을 위한 random seed 설정
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


# 최대 시퀀스 길이 설정 (메모리 효율을 위해 적절히 조절)
max_seq_length = 1024

# =====================================================
# 🔑 QLoRA의 "Q" 부분: 4-bit 양자화 모델 로드
# =====================================================
# "bnb-4bit"가 바로 QLoRA의 핵심!
# - 일반 모델: FP16 (2 bytes/param) → 1B 모델 = ~2GB
# - 4-bit 양자화: NF4 (0.5 bytes/param) → 1B 모델 = ~0.5GB
# 이 덕분에 16GB VRAM으로도 대형 모델 파인튜닝이 가능합니다.
# =====================================================
model_name = "unsloth/gemma-3-1b-it-unsloth-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
)

print(f"✅ 4-bit 양자화 모델 로드 완료: {model_name}")
print(f"   → 이것이 QLoRA의 'Q' (Quantized) 부분입니다!")

# =====================================================
# 📊 파인튜닝 전후 비교를 위한 베이스 모델 응답 저장
# =====================================================
# 학습 전 베이스 모델의 응답을 저장해두면, 학습 후 비교 가능!
# 학습에 사용하지 않는 데이터(test_samples)로 테스트합니다.

from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# Chat template 적용 (추론에 필요)
tokenizer = get_chat_template(tokenizer, chat_template="gemma3")

# 테스트용 데이터 로드 (학습에 사용하지 않는 인덱스 5000~5004)
test_dataset = load_dataset("Thytu/ChessInstruct", split="train[5000:5005]")
print(f"\n📋 테스트 데이터 {len(test_dataset)}개 로드 완료 (학습에 미사용)")

# 베이스 모델 응답 저장용 리스트
base_model_responses = []

# 재현성을 위한 seed 설정
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("\n🔍 학습 전 베이스 모델 응답 수집 중...")
for i, sample in enumerate(test_dataset):
    # 테스트 메시지 구성
    test_messages = [
        {"role": "system", "content": sample["task"]},
        {"role": "user", "content": sample["input"]}
    ]
    
    # Chat template 적용
    text = tokenizer.apply_chat_template(
        test_messages,
        tokenize=False,
        add_generation_prompt=True,
    ).removeprefix('<bos>')
    
    # 추론 실행 (스트리밍 없이)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # 응답 추출 (프롬프트 제외)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    base_model_responses.append({
        "question": sample["input"],
        "expected": sample["expected_output"],
        "base_response": response.strip()
    })
    print(f"   [{i+1}/{len(test_dataset)}] 베이스 응답 저장 완료")

print(f"\n✅ 베이스 모델 응답 {len(base_model_responses)}개 저장 완료!")
print("   → 학습 후 이 응답들과 비교하여 파인튜닝 효과를 확인합니다.")

---

# Step 2: Full Fine-tuning 시도 (문제 체감)

Full Fine-tuning의 한계를 직접 체감합니다.

### Concept Check: Full Fine-tuning이란?

**Full Fine-tuning**은 사전학습된 모델의 **모든 파라미터**를 업데이트하는 방식입니다.

**문제점:**
- 모든 파라미터의 gradient를 저장해야 하므로 **메모리 사용량이 매우 큼**
- 대형 모델(1B+ 파라미터)의 경우 일반 GPU로는 학습 불가능

**메모리 요구량 계산 (1B 모델 기준):**
```
모델 파라미터: 1B × 4 bytes (FP32) = 4GB
Gradient: 1B × 4 bytes = 4GB
Optimizer states (AdamW): 1B × 8 bytes = 8GB (momentum + variance)
Activations (배치 크기에 따라): ~4-16GB

총 필요 메모리: 약 20-32GB (FP32 기준)
```

16GB VRAM 환경에서는 **Full Fine-tuning이 사실상 불가능**합니다.

### Concept Check: 비트 정밀도(Bit Precision) — 4bit, 8bit, 16bit, 32bit의 차이

Full Fine-tuning의 메모리 문제를 이해하려면 먼저 **비트 정밀도**가 무엇인지 알아야 합니다.

---

#### 1. 비트 정밀도란?

**비트 정밀도(Bit Precision)**는 숫자 하나를 저장하는 데 사용하는 비트 수입니다. 딥러닝에서 모델의 가중치(weight)와 활성화(activation)를 저장하는 방식을 결정합니다.

| 정밀도 | 비트 수 | 파라미터당 메모리 | 표현 범위 | 주요 용도 |
|--------|---------|-----------------|-----------|----------|
| **FP32** | 32-bit | 4 bytes | ±3.4×10³⁸ | 학습 기본값, Optimizer |
| **FP16/BF16** | 16-bit | 2 bytes | ±65,504 / ±3.4×10³⁸ | Mixed Precision 학습 |
| **INT8** | 8-bit | 1 byte | -128 ~ 127 | 추론 최적화 |
| **NF4** | 4-bit | 0.5 bytes | 16단계 | QLoRA (극한 효율) |

---

#### 2. 각 정밀도의 특징

**FP32 (Full Precision)**
- 가장 정확하지만 **메모리를 가장 많이 사용**
- 전통적인 딥러닝 학습의 기본값
- Optimizer (AdamW)의 momentum, variance는 반드시 FP32로 저장

**FP16/BF16 (Half Precision)**
- FP32 대비 **메모리 50% 절감**
- FP16: 작은 수 표현에 강점 (CV, NLP 일반)
- BF16: 큰 수 표현에 강점 (LLM, Transformer 권장)
- **Mixed Precision**: FP16 연산 + FP32 accumulator로 정확도 유지

**INT8 (8-bit Integer)**
- **메모리 75% 절감** (FP32 대비)
- 추론(Inference)에 주로 사용
- 학습 시에는 gradient 정확도 문제로 제한적
- bitsandbytes의 `adamw_8bit` 옵티마이저가 여기 해당

**NF4 (4-bit NormalFloat)**
- **메모리 87.5% 절감** (FP32 대비)
- QLoRA의 핵심 기술
- 가중치 분포에 최적화된 비선형 양자화
- 일반 INT4보다 정보 손실이 적음

---

#### 3. Trade-off: 정밀도 vs 메모리 vs 성능

```
정밀도 높음 ◀──────────────────────────────▶ 정밀도 낮음
   FP32          FP16/BF16        INT8          NF4
    │               │              │             │
    ▼               ▼              ▼             ▼
 메모리 大        메모리 中       메모리 小     메모리 極小
 정확도 高        정확도 高       정확도 中     정확도 中~低
 학습 가능        학습 가능       추론 전용*    추론 전용*
                                              (*QLoRA로 학습)
```

**핵심 Trade-off:**
| 항목 | 고정밀도 (FP32) | 저정밀도 (4-bit) |
|------|----------------|------------------|
| **메모리** | 많이 사용 | 적게 사용 |
| **학습 속도** | 느림 | 빠름 |
| **수치 정확도** | 높음 | 낮음 (근사) |
| **모델 성능** | 최적 | 약간 손실 가능 |
| **학습 가능성** | Full FT 가능 | Adapter만 학습 |

> **실무 결론**: 4-bit 양자화는 성능 손실이 1-3% 수준으로 매우 작고, 메모리는 8배 절감됩니다. 이것이 QLoRA가 인기 있는 이유입니다.

---

#### 4. 실제 수치로 보는 메모리 차이

**1B (10억) 파라미터 모델 기준:**

| 정밀도 | 모델 메모리 | 7B 모델 환산 | 16GB로 가능? |
|--------|------------|-------------|-------------|
| FP32 | **4.0 GB** | 28 GB | ❌ 불가능 |
| FP16 | **2.0 GB** | 14 GB | ⚠️ 추론만 |
| INT8 | **1.0 GB** | 7 GB | ⚠️ 추론만 |
| NF4 | **0.5 GB** | 3.5 GB | ✅ QLoRA 학습 가능 |

**Full Fine-tuning에 필요한 총 메모리 (FP16 Mixed Precision):**
```
7B 모델 Full FT 예상 메모리:
├─ 모델 가중치 (FP16):    14 GB
├─ Gradient (FP16):       14 GB
├─ Optimizer (FP32):      28 GB (momentum + variance)
├─ Activations:           ~10-20 GB
└─ 총합:                  66-76 GB  ← A100 80GB 필요!

vs QLoRA (4-bit base + LoRA):
├─ 모델 가중치 (NF4):     3.5 GB
├─ LoRA Gradient:         ~0.1 GB (1% 파라미터만)
├─ Optimizer:             ~0.2 GB
├─ Activations:           ~4-8 GB
└─ 총합:                  8-12 GB  ← RTX 4090/5060ti 가능!
```

---

#### 5. 언제 어떤 정밀도를 선택할까?

| 상황 | 권장 정밀도 | 이유 |
|------|-----------|------|
| **학습 (충분한 GPU)** | FP16/BF16 + FP32 Optimizer | 속도와 정확도 균형 |
| **학습 (제한된 GPU)** | **NF4 + LoRA (QLoRA)** | 8배 메모리 절감 |
| **추론 (속도 중요)** | INT8 또는 NF4 | 빠른 응답 시간 |
| **추론 (정확도 중요)** | FP16 | 최소한의 손실 |

> **이 실습에서는**: NF4 (4-bit) base model + FP16 LoRA adapter = **QLoRA** 방식을 사용합니다!

### TODO 1: Full Fine-tuning 메모리 요구량 계산 및 시각화

- **요구사항**:
  1. 모델의 총 파라미터 수를 계산하고 Full Fine-tuning에 필요한 예상 메모리를 출력하세요.
  2. 비트 정밀도별 메모리 사용량을 시각화하여 QLoRA의 필요성을 이해하세요.
  3. 7B 모델 기준 Full FT 메모리 요구량을 계산하여 16GB VRAM으로는 불가능함을 확인하세요.
- **입력**: model 객체
- **출력**: 총 파라미터 수, 예상 Full FT 메모리 (GB), 비트별 메모리 시각화 그래프
- **힌트**:
  - `model.parameters()`를 사용하여 파라미터를 순회할 수 있습니다.
  - `total_params * bytes_per_param / (1024**3)`로 GB 단위로 변환합니다.

> **참고**: 이 실습에서는 **메모리 효율**을 위해 1B 모델을 로드하지만, **실제 PEFT의 필요성**은 7B+ 모델에서 더욱 명확합니다. 따라서 메모리 계산은 **7B 모델 기준**으로 수행하여 "왜 QLoRA가 필요한가?"를 명확히 이해합니다.

In [ ]:
# TODO: Full Fine-tuning 메모리 요구량 계산 및 시각화

import matplotlib.pyplot as plt
import numpy as np

# =====================================================
# 📊 비트 정밀도별 메모리 사용량 비교
# =====================================================
# 왜 4-bit 양자화가 필요한지 숫자로 명확히 이해합니다.

def compare_precision_memory(num_params=7_000_000_000):
    """7B 파라미터 모델 기준 비트별 메모리 비교"""
    precisions = {
        'FP32\n(32-bit)': 4.0,      # 4 bytes/param
        'FP16/BF16\n(16-bit)': 2.0, # 2 bytes/param
        'INT8\n(8-bit)': 1.0,       # 1 byte/param
        'NF4\n(4-bit)': 0.5,        # 0.5 bytes/param
    }
    
    print("=" * 60)
    print("📊 7B 모델 기준 비트별 메모리 사용량")
    print("=" * 60)
    
    memories = []
    for name, bytes_per_param in precisions.items():
        memory_gb = (num_params * bytes_per_param) / (1024**3)
        memories.append(memory_gb)
        clean_name = name.replace('\n', ' ')
        print(f"{clean_name:20} → {memory_gb:.2f} GB")
    
    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 왼쪽: 비트별 메모리 비교
    colors = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']  # 빨강→초록
    bars = axes[0].bar(list(precisions.keys()), memories, color=colors, edgecolor='black')
    axes[0].axhline(y=16, color='red', linestyle='--', linewidth=2, label='RTX 5060ti VRAM (16GB)')
    axes[0].set_ylabel('Memory Usage (GB)', fontsize=12)
    axes[0].set_title('7B Model: Memory Usage by Bit Precision', fontsize=14, fontweight='bold')
    axes[0].legend(loc='upper right')
    
    # 막대 위에 값 표시
    for bar, mem in zip(bars, memories):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                     f'{mem:.1f}GB', ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # 오른쪽: Full Fine-tuning 메모리 breakdown
    fp16_model = 2.0   # 모델 가중치 (FP16)
    fp16_grad = 2.0    # Gradient
    fp16_optim = 4.0   # Optimizer (momentum + variance)
    fp16_activation = 4.0  # Activations (추정)
    
    components = ['Model\nWeights', 'Gradients', 'Optimizer\nStates', 'Activations\n(est.)']
    values = [fp16_model, fp16_grad, fp16_optim, fp16_activation]
    colors2 = ['#3498DB', '#E74C3C', '#9B59B6', '#F39C12']
    
    bars2 = axes[1].bar(components, values, color=colors2, edgecolor='black')
    axes[1].axhline(y=16, color='red', linestyle='--', linewidth=2, label='VRAM Limit (16GB)')
    
    # 누적 합계선
    cumsum = np.cumsum(values)
    for i, (bar, val, cum) in enumerate(zip(bars2, values, cumsum)):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val:.1f}GB', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # 총합 표시
    total = sum(values)
    axes[1].axhline(y=total, color='purple', linestyle=':', linewidth=2, alpha=0.7)
    axes[1].text(3.5, total + 0.3, f'Total: {total:.1f}GB', fontsize=11, fontweight='bold', color='purple')
    
    axes[1].set_ylabel('Memory Usage (GB)', fontsize=12)
    axes[1].set_title('Full Fine-tuning (FP16) Memory Breakdown', fontsize=14, fontweight='bold')
    axes[1].legend(loc='upper left')
    axes[1].set_ylim(0, 16)
    
    plt.tight_layout()
    plt.show()
    
    return precisions, memories

def calculate_full_ft_memory(model):
    """Full Fine-tuning에 필요한 예상 메모리를 계산합니다."""
    # TODO: 총 파라미터 수를 계산하세요 (힌트: sum(p.numel() for p in model.parameters()))
    total_params = None
    
    # FP16 기준 Full FT 메모리 계산
    # 모델 가중치: 2 bytes per param (FP16)
    # Gradient: 2 bytes per param (FP16)
    # Optimizer (AdamW): 8 bytes per param (FP32 optimizer states)
    # Activations: 모델 × 2 (보수적 추정)
    
    # TODO: 각 구성요소별 메모리를 GB 단위로 계산하세요
    model_memory = None
    grad_memory = None
    optim_memory = None
    activation_memory = None
    
    # TODO: 총 필요 메모리를 계산하세요
    total_memory = None
    
    return total_params, {
        'model': model_memory,
        'gradient': grad_memory, 
        'optimizer': optim_memory,
        'activation': activation_memory,
        'total': total_memory
    }

# 실행
print("\n" + "=" * 60)
print("🔥 Full Fine-tuning이 왜 어려운가?")
print("=" * 60)

# 1. 비트별 메모리 비교 시각화
precisions, memories = compare_precision_memory()

# 2. 현재 모델의 Full FT 메모리 요구량 계산
total_params, memory_breakdown = calculate_full_ft_memory(model)

print("\n" + "=" * 60)
print(f"📊 현재 모델 ({total_params/1e9:.2f}B params) Full FT 메모리 분석")
print("=" * 60)
print(f"  • 모델 가중치 (FP16):     {memory_breakdown['model']:.2f} GB")
print(f"  • Gradient (FP16):        {memory_breakdown['gradient']:.2f} GB")
print(f"  • Optimizer States (FP32): {memory_breakdown['optimizer']:.2f} GB")
print(f"  • Activations (추정):     {memory_breakdown['activation']:.2f} GB")
print(f"  ─────────────────────────────────")
print(f"  • 총 필요 메모리:         {memory_breakdown['total']:.2f} GB")
print(f"\n  💡 현재 모델({total_params/1e9:.2f}B)은 16GB에 들어갈 수 있지만,")
print(f"     7B 모델은 약 {memory_breakdown['total']*7:.0f}GB가 필요하여 사실상 불가능합니다!")
print(f"  ❌ 또한 현재 4-bit 양자화 모델은 gradient 계산 자체가 불가능합니다!")
print(f"  ✅ 해결책: PEFT (LoRA) + 4-bit 양자화 = QLoRA")

# 3. 왜 4-bit 양자화 모델에서 Full FT가 안 되는지 설명
print("\n" + "=" * 60)
print("⚠️ 참고: 현재 로드된 모델은 4-bit 양자화 모델입니다")
print("=" * 60)
print("  • 4-bit 양자화 가중치는 gradient 계산이 불가능합니다.")
print("  • requires_grad=True 설정 시 RuntimeError 발생")
print("  • 이것이 바로 QLoRA가 LoRA adapter만 학습하는 이유입니다!")
print("  • Base model(4-bit)은 고정 ❄️, LoRA adapter(FP16)만 학습 🔥")

### Test: Full Fine-tuning 메모리 문제 이해

위 시각화를 통해 다음을 확인할 수 있습니다:

1. **비트별 메모리 비교** (왼쪽 그래프)
   - FP32: 4GB (1B 모델 기준)
   - FP16: 2GB → 여전히 학습 시 추가 메모리 필요
   - INT8: 1GB
   - **NF4 (4-bit): 0.5GB** → 양자화의 위력!

2. **Full FT 메모리 구성** (오른쪽 그래프)
   - 모델 가중치 + Gradient + Optimizer + Activations
   - 1B 모델은 약 12GB 필요 → 16GB VRAM으로 빠듯하지만 가능성 있음
   - 하지만 7B 모델은 **66-76GB 필요** → A100 80GB급 GPU 필요
   - 현재 로드된 4-bit 양자화 모델은 gradient 계산 자체가 불가능

3. **4-bit 양자화 모델의 특성**
   - 양자화된 가중치는 gradient 계산 불가
   - 이것이 QLoRA의 핵심: Base model 고정 + LoRA만 학습

> **체감**: "숫자로 보니 Full FT가 왜 안 되는지 확실히 이해됐다! QLoRA가 필요한 이유가 있구나"

---

# Step 3: PEFT/LoRA 원리 이해

Full Fine-tuning의 메모리 문제를 해결하는 PEFT와 LoRA에 대해 알아봅니다.

### Concept Check: PEFT가 필요한 이유

**Step 2에서 확인한 문제:**
- Full Fine-tuning은 모든 파라미터를 학습 → 메모리 폭발
- 16GB VRAM으로는 1B 모델도 Full FT 불가능

**해결책: PEFT (Parameter-Efficient Fine-Tuning)**
- 전체 파라미터 중 **일부만 학습**
- 사전학습된 가중치는 **고정(freeze)**
- 적은 수의 **추가 파라미터(adapter)**만 학습

**장점:**
- 메모리 사용량 대폭 감소
- 학습 시간 단축
- Full FT와 비슷한 성능 달성 가능

### Concept Check: LoRA의 원리

**LoRA (Low-Rank Adaptation)** 는 가장 널리 사용되는 PEFT 기법입니다.

**핵심 아이디어:**
- 기존 가중치 행렬 $W \in \mathbb{R}^{d \times d}$를 직접 업데이트하지 않음
- 대신 저차원 행렬 $B \in \mathbb{R}^{d \times r}, A \in \mathbb{R}^{r \times d}$를 학습 (r << d)
- 출력: $h = Wx + BAx$ (원본 가중치 + adapter)

```
┌─────────────────────────────────────────┐
│         Pretrained Weights (W)          │ ← 고정 (freeze) ❄️
│              d × d                      │
└─────────────────────────────────────────┘
                    +
┌──────────┐   ┌──────────┐
│    B     │ × │    A     │  ← 학습 (trainable) 🔥
│  d × r   │   │  r × d   │
└──────────┘   └──────────┘
        ↓           ↓
   (d×r) × (r×d) = (d×d) ← W와 같은 차원!
```

---

#### 행렬 연산 상세 설명

**수식 $h = Wx + BAx$의 계산 순서:**

입력 벡터 $x \in \mathbb{R}^{d \times 1}$에 대해:

| 단계 | 연산 | 차원 변화 | 설명 |
|------|------|----------|------|
| 1 | $x$ | $(d \times 1)$ | 입력 벡터 |
| 2 | $Ax$ | $(r \times d) \times (d \times 1) = (r \times 1)$ | **차원 축소** (d → r) |
| 3 | $B(Ax)$ | $(d \times r) \times (r \times 1) = (d \times 1)$ | **차원 복원** (r → d) |
| 4 | $Wx + BAx$ | $(d \times 1) + (d \times 1) = (d \times 1)$ | 원본 출력 + adapter 출력 |

**가중치 업데이트 관점에서 $BA$:**
$$BA = (d \times r) \times (r \times d) = (d \times d)$$

→ $W$와 같은 차원이므로 $W' = W + BA$로 병합 가능!

**왜 이렇게 설계했을까?**
- $r \ll d$이므로 $A$와 $B$의 파라미터 수는 $2 \times r \times d$
- 원본 $W$의 파라미터 수 $d \times d$에 비해 **약 $\frac{2r}{d}$ 배** (r=8, d=4096이면 약 0.4%)
- 저차원(r)을 거쳐가면서 **저차원 부분공간에서의 변화만 학습** → 효율적!

---

**파라미터 효율성:**
- 기존: $d \times d$ = 수백만~수억 개
- LoRA: $2 \times r \times d$ (r=8이면 원본의 약 1%)

> **체감**: "1% 파라미터만 학습하면 되는구나! 메모리 문제가 해결되겠네"

### Concept Check: LoRA 핵심 하이퍼파라미터

| 파라미터 | 설명 | 권장값 |
|----------|------|--------|
| **r (rank)** | 저차원 행렬의 차원. 클수록 표현력↑, 메모리↑ | 8, 16, 32, 64 |
| **lora_alpha** | 스케일링 값. 출력에 α/r을 곱함. 클수록 adapter 영향↑ | r × 2 (예: r=8이면 alpha=16) |
| **target_modules** | LoRA를 적용할 레이어 | q_proj, k_proj, v_proj, o_proj 등 |
| **lora_dropout** | 드롭아웃 비율. 과적합 방지 | 0.0 ~ 0.1 |

**target_modules 선택:**
- **Attention**: q_proj, k_proj, v_proj, o_proj
- **MLP**: gate_proj, up_proj, down_proj
- 일반적으로 Attention 레이어에 적용하는 것이 효과적

---

# Step 4: LoRA 적용 및 학습

LoRA를 적용하여 실제 학습을 수행합니다.

### Concept Check: Chat Template

LLM 파인튜닝에서는 입력 데이터를 모델이 이해할 수 있는 **chat 형식**으로 변환해야 합니다.

**Gemma-3 Chat Template:**
```
<bos><start_of_turn>user
질문 내용<end_of_turn>
<start_of_turn>model
답변 내용<end_of_turn>
```

Unsloth는 `get_chat_template` 함수로 다양한 템플릿을 지원합니다.

### TODO 2: LoRA 설정값 채우기

- **요구사항**: LoRA 핵심 하이퍼파라미터를 설정하세요.
- **입력**: 없음
- **출력**: LoRA가 적용된 model 객체
- **힌트**: r=8, lora_alpha=16 (r×2), target_modules는 Attention 레이어

In [ ]:
# TODO: LoRA 설정값 채우기

# TODO: rank 값을 설정하세요 (8, 16, 32, 64 중 선택)
r = None

# TODO: r * 2 권장
lora_alpha = None

# TODO: LoRA를 적용할 레이어 리스트를 설정하세요
target_modules = None

# TODO: 0을 권장
lora_dropout = None

bias = "none"

# LoRA 적용
model = FastModel.get_peft_model(
    model,
    r=r,
    target_modules=target_modules,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias=bias,
)

print(f"LoRA 적용 완료!")
print(f"- r (rank): {r}")
print(f"- lora_alpha: {lora_alpha}")
print(f"- target_modules: {target_modules}")

### Guided Build: 데이터셋 로드 및 Chat Template 적용

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

# Note: tokenizer에 chat template이 이미 적용되어 있음 (Step 1에서 적용)
# 만약 Step 1을 건너뛴 경우를 위해 아래 코드가 있지만, 
# 정상 흐름에서는 Step 1에서 이미 적용됨

# 학습용 데이터셋 로드 (인덱스 0~4999, 테스트용과 분리!)
# - 학습 데이터: train[:5000]
# - 테스트 데이터: train[5000:5005] (Step 1에서 이미 로드됨)
dataset = load_dataset("Thytu/ChessInstruct", split="train[:5000]")
print(f"학습 데이터셋 크기: {len(dataset)}")
print(f"샘플 데이터: {dataset[0]}")
print(f"\n⚠️ 테스트 데이터(5000~5004)는 학습에서 제외됨 → 공정한 비교 가능")

### Concept Check: 데이터 변환 패턴 이해

TODO 3에서는 원본 데이터를 **chat 형식(conversations)**으로 변환하는 함수를 작성합니다. 먼저 변환 패턴을 이해해봅시다.

---

#### 원본 데이터 구조

ChessInstruct 데이터셋의 각 샘플은 다음 3개 필드로 구성됩니다:

| 필드 | 설명 | 예시 |
|------|------|------|
| `task` | 수행할 작업 설명 | "Predict the next move in the chess game..." |
| `input` | 체스 기보 (입력) | "1. e4 e5 2. Nf3 Nc6 3. Bb5" |
| `expected_output` | 예상 다음 수 (정답) | "a6" |

---

#### conversations 형식 변환 결과

LLM 파인튜닝을 위해 위 데이터를 **system/user/assistant 역할** 형식으로 변환합니다:

```python
{
    "conversations": [
        {"role": "system", "content": task},       # 작업 지시사항
        {"role": "user", "content": input},        # 사용자 입력
        {"role": "assistant", "content": expected_output}  # 모델 응답
    ]
}
```

---

#### 1개 샘플 변환 예시

**변환 전 (원본):**
```python
{
    "task": "Predict the next move in this chess game.",
    "input": "1. e4 e5 2. Nf3 Nc6",
    "expected_output": "Bb5"
}
```

**변환 후 (conversations):**
```python
{
    "conversations": [
        {"role": "system", "content": "Predict the next move in this chess game."},
        {"role": "user", "content": "1. e4 e5 2. Nf3 Nc6"},
        {"role": "assistant", "content": "Bb5"}
    ]
}
```

> **핵심**: `task` → system, `input` → user, `expected_output` → assistant로 매핑!

### TODO 3: convert_to_chatml 함수 작성

- **요구사항**: 데이터셋을 chat 형식(conversations)으로 변환하는 함수를 완성하세요.
- **입력**: example (데이터셋의 한 행)
- **출력**: {"conversations": [{"role": ..., "content": ...}, ...]}
- **힌트**: system(task), user(input), assistant(expected_output) 역할로 구성

In [ ]:
# TODO: convert_to_chatml 함수 작성

# TODO: conversations 리스트를 완성하세요
# 각 딕셔너리는 "role"과 "content" 키를 가져야 합니다
# role: "system", "user", "assistant" 중 하나
# content: example의 적절한 필드 값
def convert_to_chatml(example):
    """데이터셋을 chat 형식으로 변환합니다."""
    return {
        "conversations": [
            # TODO: system, user, assistant 역할의 메시지를 작성하세요
        ]
    }

# 데이터셋 변환
dataset = dataset.map(convert_to_chatml)
print(f"변환된 샘플: {dataset[0]['conversations']}")

### Guided Build: 텍스트 포맷팅 및 Trainer 설정

In [ ]:
def formatting_prompts_func(examples):
    """chat template을 적용하여 텍스트로 변환합니다."""
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix('<bos>')
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"포맷된 텍스트 예시:\n{dataset[0]['text'][:500]}...")

In [ ]:
from trl import SFTConfig, SFTTrainer

# 학습 설정 (16GB VRAM 최적화)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        output_dir="outputs",
        per_device_train_batch_size=2,      # 메모리에 맞게 조절
        gradient_accumulation_steps=4
        ,       # 실효 배치 크기 = 4 × 2 = 8
        learning_rate=5e-5,
        max_steps=100,                       # 빠른 실습을 위해 100 step만
        logging_steps=10,
        fp16=False,                           # 16bit 학습으로 메모리 절약
        bf16=True,
        optim="adamw_8bit",                  # Unsloth 8bit 최적화
        report_to="none",
    ),
)

print("Trainer 설정 완료!")

In [ ]:
from unsloth.chat_templates import train_on_responses_only

# Response만 학습하도록 설정 (instruction 부분은 무시)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print("Response-only 학습 설정 완료!")

### Test: 학습 진행 및 메모리 확인

In [ ]:
# 학습 전 메모리 상태
if torch.cuda.is_available():
    start_memory = torch.cuda.max_memory_reserved() / 1024**3
    print(f"학습 전 메모리: {start_memory:.2f} GB")

# 학습 시작
print("\n🚀 학습 시작!")
trainer_stats = trainer.train()

# 학습 후 메모리 상태
if torch.cuda.is_available():
    end_memory = torch.cuda.max_memory_reserved() / 1024**3
    print(f"\n학습 후 메모리: {end_memory:.2f} GB")
    print(f"학습에 사용된 메모리: {end_memory - start_memory:.2f} GB")

print(f"\n✅ 학습 완료! 16GB VRAM으로도 학습이 가능합니다!")

---

# Step 5: 결과 확인 및 저장

파인튜닝된 모델을 저장하고 추론을 수행합니다.

### Concept Check: LoRA Adapter 저장 방식

LoRA 모델을 저장하는 방법은 두 가지가 있습니다:

1. **Adapter만 저장** (권장)
   - LoRA adapter 가중치만 저장 (수 MB)
   - 추론 시 base model + adapter 로드
   - 저장 용량 작음

2. **Merged 저장**
   - base model과 adapter를 병합하여 저장
   - 추론 시 단일 모델 로드
   - 저장 용량 큼 (원본 모델 크기)

**추론 시 merge:**
- adapter를 base model에 병합하면 추론 속도 향상
- `model.merge_and_unload()` 사용

### Guided Build: 모델 저장

### Concept Check: 추론을 위한 입력 구성

TODO 4에서는 학습된 모델로 추론을 수행합니다. **학습**과 **추론**에서 `apply_chat_template()` 사용 방식이 다르다는 점을 이해해야 합니다.

---

#### 학습 vs 추론 시 `apply_chat_template()` 차이점

| 구분 | `add_generation_prompt` | 설명 |
|------|------------------------|------|
| **학습** | `False` | assistant 응답이 이미 포함됨 → 생성 프롬프트 불필요 |
| **추론** | `True` | assistant 응답을 모델이 생성해야 함 → 생성 프롬프트 필요 |

---

#### `add_generation_prompt` 파라미터 설명

이 파라미터는 모델이 응답을 생성하도록 유도하는 **시작 토큰**을 추가할지 결정합니다.

**`add_generation_prompt=False` (학습 시):**
```
<start_of_turn>user
1. e4 e5 2. Nf3 Nc6<end_of_turn>
<start_of_turn>model
Bb5<end_of_turn>
```
→ assistant 응답(Bb5)이 이미 있으므로 생성 프롬프트 불필요

**`add_generation_prompt=True` (추론 시):**
```
<start_of_turn>user
1. e4 e5 2. Nf3 Nc6<end_of_turn>
<start_of_turn>model
```
→ `<start_of_turn>model` 이후 모델이 응답을 생성

---

#### 추론용 메시지 구성 예시

```python
# 추론 시에는 system + user 메시지만 구성 (assistant는 모델이 생성)
test_messages = [
    {"role": "system", "content": "Predict the next move in this chess game."},
    {"role": "user", "content": "1. e4 e5 2. Nf3 Nc6"}
]

# Chat template 적용 (추론용)
text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,  # ← 추론 시 반드시 True!
)
```

> **핵심**: 추론 시에는 `add_generation_prompt=True`로 설정하여 모델이 응답을 생성하도록 유도!

In [ ]:
# LoRA adapter만 저장 (권장)
output_dir = "gemma3-lora-chess"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"LoRA adapter 저장 완료: {output_dir}")

### TODO 4: 추론을 위한 instruction/response part 채우기

- **요구사항**: chat template의 instruction_part와 response_part를 확인하고, 테스트 입력을 구성하세요.
- **입력**: 테스트 데이터 (system, user 메시지)
- **출력**: 모델의 추론 결과
- **힌트**: Gemma3 템플릿의 시작 토큰을 확인하세요.

In [ ]:
# TODO: 추론 테스트 (학습에 사용하지 않은 테스트 데이터 활용)

# 테스트 입력 구성 (Step 1에서 저장한 test_dataset 사용)
# 학습에 사용하지 않은 데이터로 테스트해야 공정한 평가 가능!
test_sample = test_dataset[0]  # 첫 번째 테스트 샘플

# TODO: 테스트 입력 구성 - system과 user 역할의 메시지 리스트를 만드세요
# 힌트: {"role": "system", "content": ...}, {"role": "user", "content": ...}
test_messages = [
    # TODO: system 역할의 메시지 추가
    # TODO: user 역할의 메시지 추가
]

# Chat template 적용
text = tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,  # 추론 시 True로 설정
).removeprefix('<bos>')

print("입력 텍스트:")
print(text[:500] + "...")

In [ ]:
from transformers import TextStreamer

# 추론 실행
print("\n모델 응답:")
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=128,
    temperature=0.7,
    top_p=0.95,
    top_k=64,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

### 📊 파인튜닝 전후 비교: LoRA가 정말 효과가 있을까?

Step 1에서 저장해둔 **베이스 모델 응답**과 **파인튜닝 후 응답**을 비교하여 LoRA 학습의 효과를 명확히 확인합니다.

**비교 포인트:**
- 학습에 **사용하지 않은 테스트 데이터**로 평가 (공정한 비교)
- 베이스 모델 vs 파인튜닝 모델의 **응답 품질** 차이
- 체스 도메인 지식의 **실제 향상** 여부 확인

In [ ]:
# 재현성을 위한 seed 설정 (학습 전과 동일한 조건)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# 파인튜닝 전후 비교 실행
print("=" * 70)
print("📊 LoRA 파인튜닝 전후 비교 결과")
print("=" * 70)

# 파인튜닝된 모델로 테스트 데이터에 대한 응답 생성
for i, sample in enumerate(test_dataset):
    print(f"\n{'─' * 70}")
    print(f"🎯 테스트 샘플 {i+1}/{len(test_dataset)}")
    print(f"{'─' * 70}")
    
    # 질문 출력
    print(f"\n[📝 질문]")
    print(f"   Task: {sample['task'][:80]}...")
    print(f"   Input: {sample['input'][:100]}...")
    
    # 학습 전 응답 (Step 1에서 저장)
    print(f"\n[❌ 학습 전 베이스 모델 응답]")
    print(f"   {base_model_responses[i]['base_response'][:200]}...")
    
    # 학습 후 응답 생성
    test_messages = [
        {"role": "system", "content": sample["task"]},
        {"role": "user", "content": sample["input"]}
    ]
    text = tokenizer.apply_chat_template(
        test_messages,
        tokenize=False,
        add_generation_prompt=True,
    ).removeprefix('<bos>')
    
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            temperature=0.7,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    finetuned_response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], 
        skip_special_tokens=True
    ).strip()
    
    print(f"\n[✅ 학습 후 파인튜닝 모델 응답]")
    print(f"   {finetuned_response[:200]}...")
    
    # 정답 출력
    print(f"\n[🎯 정답 (expected_output)]")
    print(f"   {sample['expected_output'][:200]}...")

print(f"\n{'=' * 70}")
print("📊 비교 완료!")
print("=" * 70)
print("\n💡 관찰 포인트:")
print("   1. 학습 전: 체스 기보(notation) 해석이 어려움")
print("   2. 학습 후: 체스 도메인 지식이 향상되어 더 정확한 응답")
print("   3. 단 100 step 학습으로도 도메인 적응 효과 확인 가능")
print("\n✅ LoRA 파인튜닝이 실제로 작동하는 것을 체감했습니다!")

### Test: 파인튜닝 효과 확인

> **체감**: "Full FT와 비슷한 성능인데 효율은 훨씬 좋네"

- 16GB VRAM으로 학습 완료 ✅
- 학습 시간: 수 분 (Full FT 대비 훨씬 빠름)
- 학습 파라미터: 전체의 약 1% 미만

In [ ]:
# 학습된 파라미터 수 확인
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_percentage = (trainable_params / total_params) * 100

print(f"\n📊 LoRA 효율성 요약:")
print(f"- 총 파라미터: {total_params:,}")
print(f"- 학습 파라미터: {trainable_params:,}")
print(f"- 학습 비율: {trainable_percentage:.2f}%")
print(f"\n✅ 전체의 {trainable_percentage:.2f}%만 학습하여 효율적으로 파인튜닝 완료!")

---
## 트러블슈팅 가이드

실습/과제 진행 중 자주 발생하는 오류와 해결 방법입니다.

### GPU / CUDA 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `CUDA out of memory` | GPU 메모리 부족 | 커널 재시작 후 불필요한 모델 제거, `torch.cuda.empty_cache()` |
| `CUDA not available` | CUDA 드라이버 미설치 | `nvidia-smi` 확인 후 드라이버 재설치 |
| `bitsandbytes` 설치 오류 | CUDA 버전 불일치 | `pip install bitsandbytes --prefer-binary` |

### 모델 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `OSError: Can't load tokenizer` | 모델 접근 권한 없음 | HuggingFace 토큰 설정 확인 (`huggingface-cli login`) |
| 모델 다운로드 중단 | 네트워크 불안정 | `resume_download=True` 옵션 사용 |
| `OutOfMemoryError` (시스템 RAM) | 시스템 메모리 부족 | 다른 프로세스 종료 또는 배치 크기 축소 |

### 패키지 관련

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `ModuleNotFoundError` | 패키지 미설치 | Step 1의 `%pip install` 셀 재실행 |
| `ImportError: cannot import name` | 버전 불일치 | `%pip install --upgrade <패키지>` |

### 일반

| 증상 | 원인 | 해결 방법 |
|------|------|----------|
| `NameError: name 'xxx' is not defined` | 이전 셀 미실행 | Step 1부터 순서대로 재실행 |
| 학습 시 loss가 줄지 않음 | 학습률/에폭 설정 문제 | `learning_rate`, `num_train_epochs` 조정 |


---

## 학생용 자가 체크리스트

- [ ] Full Fine-tuning의 메모리 문제를 이해했는가?
- [ ] LoRA의 원리(저차원 분해)를 설명할 수 있는가?
- [ ] LoRA 하이퍼파라미터(r, lora_alpha, target_modules)의 의미를 이해했는가?
- [ ] Unsloth를 사용하여 QLoRA 학습을 수행할 수 있는가?
- [ ] 파인튜닝된 모델을 저장하고 추론할 수 있는가?

---

### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 삼성 청년 SW아카데미의 컨텐츠 자산으로, 보안서약서에 의거하여 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.